In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.17 Molecular Dynamics: Periodic Boundaries, Cutoffs, and Pressure

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.17",
    title="Molecular Dynamics: Periodic Boundaries, Cutoffs, and Pressure",
    blurb="Every piece of a molecular-dynamics program is already in this "
    "course, and they have never been put together. So we assemble them, "
    "and then build the layer that sits outside the integrator and that "
    "nobody warned us about: the Lennard-Jones force, a box with no "
    "walls, the minimum-image convention, a cutoff that has to be "
    "shifted or the energy will not stand still, the pressure read off "
    "the virial, honest error bars on a correlated trajectory, and "
    "Maxwell and Boltzmann emerging from nothing but Newton.",
    difficulty="advanced",
    estimate="150–190 min",
)

## Notebook overview

The reader of this volume has, by now, written every part of a
molecular-dynamics program without ever running one. Velocity Verlet
was built from scratch in [§1.6](../01-elementary-mechanics/integrators.ipynb),
with its symplecticity, its shadow Hamiltonian, and its measured
order of accuracy. The $O(N^2)$ pairwise force sum, written as one
broadcast expression with no Python loop over pairs, is
[§1.8](../01-elementary-mechanics/solar-system.ipynb), along with the
discipline of a reduced unit system. Maxwell-Boltzmann velocities,
autocorrelation functions and the Green-Kubo integral are
[§5.11](taste-of-nonequilibrium.ipynb); collisions, mean free paths
and an event-driven hard-disk gas are
[§5.12](kinetic-theory.ipynb), whose closing page hands the subject
forward in as many words. The parts are all here. Nobody has ever
switched them on together.

That is not an accident of this course. It is the shape of the
literature: the integrator is taught in the chapter on differential
equations, the sampling in the chapter on Monte Carlo, and the two
are rarely introduced to each other. Allen and Tildesley
{cite}`allen_tildesley` divide the subject the same way, and the
division is instructive: their molecular-dynamics chapter is
integrators and ensembles, while periodic boundaries and neighbour
lists wait for a later chapter, and structure and error analysis for
a later one still. So this notebook is the assembly, and what it
adds is precisely the layer that sits *outside* the integrator.

Five things are genuinely new here. We differentiate the
Lennard-Jones potential and turn it into a force, which no notebook
in this course has done ([§0.13](../00-foundations/optimization.ipynb)
used the 12-6 form only as a scalar function to hand a minimizer).
We build a box with **no walls at all** and the **minimum-image
convention** that makes it work, and we watch what happens to a
simulation that forgets it. We truncate the potential at a finite
range and discover that a cutoff must be *shifted* or the energy
will not stand still. We measure the **pressure** from the trajectory
through the virial, which this volume has never done even while
deriving a full equation of state in
[§5.15](van-der-waals.ipynb). And we start a gas from a velocity
distribution that is not Maxwell-Boltzmann and watch smooth
Newtonian flow make it Maxwell-Boltzmann, with a statistical test
rather than a glance. Along the way we build the error analysis that
a correlated trajectory demands, because a mean without an honest bar
on it is not a measurement.

Two scope decisions, both deliberate. We work in **two dimensions**,
which is a choice with precedent rather than a shortcut: Gould,
Tobochnik and Christian {cite}`gould_tobochnik` and Schroeder
{cite}`schroeder_thermal` both teach molecular dynamics in 2-D for
the stated reasons that it visualises directly, costs far less, and
demonstrates every principle the three-dimensional case does. It also
continues [§1.8](../01-elementary-mechanics/solar-system.ipynb), whose
solar system is coplanar. And we run **NVE only, with no thermostat**,
which lets us say plainly a thing the introductory literature almost
never says: velocity rescaling does not sample the canonical
ensemble, and neither does the Berendsen scheme that generalizes it.
Exercise 6 measures exactly how it fails. Frenkel and Smit
{cite}`frenkel_smit` is the standard reference for everything deeper.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

### The pair potential, and the force it implies

A neutral atom feels two things from a neighbour: a weak long-range
attraction from correlated fluctuations of the two electron clouds,
which decays as $r^{-6}$, and a violent short-range repulsion once
the clouds overlap and the Pauli principle objects. Lennard-Jones
combined them in the form that has carried simulation since Rahman
put it on a computer in 1964 {cite}`rahman1964`, choosing the
repulsive exponent as the square of the attractive one because it is
cheap to evaluate rather than because it is right:

```{math}
:label: eq-md-lj
u(r) \;=\; 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12}
  - \left(\frac{\sigma}{r}\right)^{6}\right].
```

Here $\sigma$ is the distance at which $u$ crosses zero and
$\varepsilon$ the depth of the well, whose minimum sits at
$r_{\min} = 2^{1/6}\sigma$. Everything in this notebook runs in
**reduced Lennard-Jones units**, $\varepsilon = \sigma = m = k_B = 1$,
the same move as the astronomical units of
[§1.8](../01-elementary-mechanics/solar-system.ipynb): lengths in
$\sigma$, energies in $\varepsilon$, temperature as
$T^\ast = k_BT/\varepsilon$, and time in
$\sigma\sqrt{m/\varepsilon}$. Argon fixes the scale at
$\varepsilon/k_B = 119.8\ \mathrm{K}$ and
$\sigma = 3.405\ \text{Å}$ if a physical number is wanted.

A potential is not yet a simulation. What the integrator needs is the
force, and for a central pair potential that is $-\,\mathrm{d}u/\mathrm{d}r$
along the line of centres. Differentiating {eq}`eq-md-lj` and writing
the result in the vector form the code actually wants, with
$\mathbf r_{ij} = \mathbf r_i - \mathbf r_j$ and $r = |\mathbf r_{ij}|$,

```{math}
:label: eq-md-force
\mathbf f_{ij} \;=\; \frac{24\varepsilon}{r^{2}}
  \left[2\left(\frac{\sigma}{r}\right)^{12}
  - \left(\frac{\sigma}{r}\right)^{6}\right]\mathbf r_{ij}.
```

The bracket over $r^2$ is written that way on purpose: every term
needs only $r^2$, so no square root is ever taken in the inner loop.
The force vanishes at $r_{\min}$, is repulsive (parallel to
$\mathbf r_{ij}$) inside it, and attractive outside.

### A box with no walls

A few hundred atoms in a box are almost all *surface*. In a
$10 \times 10$ square arrangement, sixty-four of the hundred atoms
touch an edge, so a simulation with walls measures the walls. The
standard repair, older than most of the field, is to abolish the
walls: the cell of side $L$ is made **periodic**, tiled infinitely in
every direction, so an atom that leaves the right edge re-enters at
the left and no atom is ever at a boundary. The bulk is then modelled
by a system with no surface at all.

Periodicity alone is not enough, and here is the part every
introduction skips over in a sentence. Two atoms near opposite edges
of the cell are, as coordinates, nearly $L$ apart; as *physics* they
are neighbours, because each sits beside a periodic image of the
other. Under the **minimum-image convention** the interaction of $i$
with $j$ is taken with the single nearest of the infinitely many
images of $j$, which for a cubic or square cell is one line of
arithmetic on each Cartesian component of the separation:

```{math}
:label: eq-md-minimage
d_\alpha \;\longleftarrow\; d_\alpha
  - L\,\mathrm{round}\!\left(\frac{d_\alpha}{L}\right),
  \qquad \alpha = x, y,
```

which returns the representative of $d_\alpha$ in $[-L/2, L/2)$.
The convention is consistent only if no atom interacts with two
images of the same partner, which is why every cutoff must satisfy
$r_c \le L/2$: a constraint that ties the range of the potential to
the size of the box, and quietly sets the cost of every simulation
ever run.

### The cutoff, and why it must be shifted

Evaluating {eq}`eq-md-lj` for every pair is $O(N^2)$ work, and beyond
a few $\sigma$ the potential is negligible: $u(2.5\sigma)$ is
$-0.0163\,\varepsilon$, under two percent of the well depth. So the
interaction is truncated at $r_c = 2.5\sigma$, the value the field has
used since Verlet {cite}`verlet1967`. Truncation alone leaves a
discontinuity of size $u(r_c)$ at the cutoff, and a discontinuous
potential is not the potential of any force: every pair that crosses
$r_c$ takes a step in the total energy, so the quantity we monitor to
certify the integrator random-walks away for reasons that have nothing
to do with the integrator. Shifting removes it:

```{math}
:label: eq-md-shift
u_{\rm ts}(r) \;=\;
\begin{cases}
u(r) - u(r_c), & r < r_c, \\[2pt]
0, & r \ge r_c,
\end{cases}
```

which is continuous at $r_c$, leaves the force unchanged everywhere
(so the trajectory is *identical*), and makes $K + U$ a genuine
constant of the motion again. This truncated-and-shifted potential is
a slightly different physical model from the full Lennard-Jones one,
and honest practice says so: every number below refers to
$u_{\rm ts}$ with $r_c = 2.5$, including the theoretical predictions
it is checked against.

### Temperature, with the correction nobody mentions

Equipartition assigns $\tfrac12 k_BT$ to each quadratic degree of
freedom, so a two-dimensional gas of $N$ atoms would seem to give
$\langle K\rangle = N k_B T$. It does not, because the total momentum
is a constant of the motion that we set to zero when the run starts
and that Newton's third law then holds there forever. Two of the
$2N$ velocity components are therefore not free, and the honest
thermometer is

```{math}
:label: eq-md-temp
k_B T \;=\; \frac{2\langle K\rangle}{2N-2}
 \;=\; \frac{1}{N-1}\left\langle \tfrac12\sum_i m|\mathbf v_i|^2\right\rangle .
```

The correction is $1/N$, invisible at $N = 10^{23}$ and worth
$1.6\%$ at the $N = 64$ of this notebook. Two cautions come with
{eq}`eq-md-temp`. It is a *time average*, meaningful only once the
system has equilibrated; the instantaneous $2K/(2N-2)$ fluctuates by
tens of percent and is not a temperature. And it is a
**microcanonical** temperature, the one belonging to a system of
fixed energy, which is why nothing in this notebook is at a fixed
temperature at all.

### Pressure from the virial

There are no walls left to push on, so pressure cannot be measured as
a wall force. It is measured instead from the *virial*, the
configuration sum that Clausius introduced for exactly this purpose.
For a pairwise potential in $d$ dimensions the result, derived in
Allen and Tildesley {cite}`allen_tildesley` and in Frenkel and Smit
{cite}`frenkel_smit`, is that the ideal-gas kinetic pressure is
corrected by the pair forces:

```{math}
:label: eq-md-virial
P V \;=\; \frac{2\langle K\rangle}{d}
  \;+\; \frac{1}{d}\Bigl\langle \textstyle\sum_{i<j}
  \mathbf r_{ij}\cdot\mathbf f_{ij}\Bigr\rangle
  \;\equiv\; \frac{2\langle K\rangle}{d} + \frac{\langle W\rangle}{d}.
```

In two dimensions $V$ is an area and $d = 2$. Repulsive pairs
($\mathbf f_{ij}$ along $+\mathbf r_{ij}$) push $W$ up and the
pressure with it; attractive pairs pull it down. That single sentence
is the whole content of the van der Waals equation of state of
[§5.15](van-der-waals.ipynb), which this volume derived analytically
and never once measured. Here we measure it. The low-density
behaviour is fixed by the **second virial coefficient**, which in two
dimensions is the plane integral

```{math}
:label: eq-md-b2
B_2(T) \;=\; -\pi\int_0^{\infty}
  \left[e^{-u(r)/k_BT} - 1\right] r\,\mathrm{d}r,
  \qquad \frac{PV}{Nk_BT} = 1 + B_2\rho + O(\rho^2),
```

computable to machine precision by quadrature and therefore an
independent judge of the simulation.

### The integrator, and what it is doing here

The stepper is velocity Verlet, kick-drift-kick, built from scratch
with its shadow Hamiltonian and its measured second-order convergence
in [§1.6](../01-elementary-mechanics/integrators.ipynb):

```{math}
:label: eq-md-verlet
\mathbf v_{n+\frac12} = \mathbf v_n + \tfrac{\Delta t}{2}\mathbf a_n,
\quad
\mathbf r_{n+1} = \mathbf r_n + \Delta t\,\mathbf v_{n+\frac12},
\quad
\mathbf v_{n+1} = \mathbf v_{n+\frac12}
  + \tfrac{\Delta t}{2}\mathbf a_{n+1}.
```

We do not rebuild it. Reusing it *is* the point: the reader wrote it
once, established there why it is the right tool for a long
Hamiltonian run, and it now arrives as an instrument, one force
evaluation per step, so that the new work can be the layer around it.

### Error bars on a trajectory

One last piece of theory, and the one the textbooks leave out. Since
[§5.8](partition-function.ipynb) the reader has known to discard the
transient at the start of a chain before averaging anything, and that
habit carries over here unchanged. What has never been priced is the
correlation that survives the cut. A molecular-dynamics run hands back
a *correlated* time series: the temperature two steps from now is very
nearly the temperature now. Successive samples are not independent, so
the familiar $\sigma/\sqrt{n}$ is not the error on the mean. The correct statement
is written with the **integrated autocorrelation time** $\tau_{\rm int}$
of the normalized autocovariance $\rho(t)$,

```{math}
:label: eq-md-tau
\tau_{\rm int} = \tfrac12 + \sum_{t=1}^{W}\rho(t),
\qquad
\sigma_{\bar x}^{2} = \frac{2\tau_{\rm int}}{n}\,\sigma_x^{2},
\qquad
n_{\rm eff} = \frac{n}{2\tau_{\rm int}},
```

so a run of $n$ samples is worth $n_{\rm eff}$ independent ones and
the naive bar is too small by $\sqrt{2\tau_{\rm int}}$. The window
$W$ cannot be taken to infinity (the tail is pure noise) and is fixed
by Sokal's self-consistent rule, the smallest $W$ with
$W \ge c\,\tau_{\rm int}(W)$ at $c = 8$. Exercise 8 builds this and
the blocking analysis that confirms it. Ambegaokar and Troyer
{cite}`ambegaokar2010` is the reference to read.

## Setup

Reduced Lennard-Jones units throughout, $\varepsilon = \sigma = m = k_B = 1$,
so masses never appear and an acceleration is a force. The data are the
state point this notebook works at (the cutoff $r_c = 2.5$, the step
$\Delta t = 0.004$, the target temperature $T = 1.0$, the reference
density $\rho = 0.4$), the series palette, and the deliberately
*non*-Maxwellian initial velocities of Exercise 7, which are the
specimen the problem hands us rather than anything to be built. The
instruments are a lattice builder and the velocity-Verlet driver of
[§1.6](../01-elementary-mechanics/integrators.ipynb), restated here so
the notebook stands alone: the driver only iterates whatever force
routine it is handed and records what that routine returns.

This notebook's own machinery is *not* here. The Lennard-Jones pair
force is Exercise 1, the minimum-image displacement Exercise 2, the
force-energy-virial machine Exercise 3, the thermometer Exercise 4, and
the correlation-time and blocking estimators Exercise 8. Every run below
is driven by the routines built in those exercises.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import quad
from scipy.stats import kstest

from ecp import animate, draw, validate

# data: the series palette.
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# data: the state point, in reduced Lennard-Jones units.
R_CUT = 2.5  # interaction cutoff, in sigma (Verlet's standing choice)
DT = 0.004  # time step, in sigma sqrt(m/epsilon)
T_TARGET = 1.0  # target temperature k_B T / epsilon
RHO = 0.4  # reference number density N / L^2, in sigma^-2


# instrument: a grid builder. Where the atoms start is not the physics —
# any non-overlapping configuration would do, and the run erases the memory
# of it within a few hundred steps.
def square_lattice(n_side, L):
    """Positions of n_side^2 atoms on a centred square lattice of side L.

    A starting configuration with no overlaps, which is all an initial
    condition has to be: the equilibration phase destroys the lattice
    long before any measurement is taken.

    Parameters
    ----------
    n_side : int
        Number of atoms along one edge; the system holds n_side^2 of them.
    L : float
        Side of the periodic cell, in sigma.

    Returns
    -------
    numpy.ndarray
        Positions, shape (n_side^2, 2), all inside [0, L).
    """
    a = L / n_side
    xs = (np.arange(n_side) + 0.5) * a
    X, Y = np.meshgrid(xs, xs, indexing="ij")
    return np.column_stack([X.ravel(), Y.ravel()])


# data: the specified non-Maxwellian initial condition of Exercise 7 — every
# atom given the SAME speed and a random direction, a delta-function shell in
# velocity space. It is the specimen the problem hands us, not machinery: the
# physics is what the dynamics does to it. The net momentum is removed, which
# is what makes the 2N-2 of eq-md-temp true for the rest of the run.
def shell_velocities(n, speed, rng):
    """n velocities of identical magnitude and uniformly random direction.

    The most emphatically non-Maxwellian velocity distribution available
    at a given kinetic energy: all the speed probability at one point.
    The mean is subtracted, so the total momentum starts at zero.

    Parameters
    ----------
    n : int
        Number of atoms.
    speed : float
        Common speed, in sigma sqrt(epsilon/m).
    rng : numpy.random.Generator
        Seeded generator supplying the directions.

    Returns
    -------
    numpy.ndarray
        Velocities, shape (n, 2), with zero net momentum.
    """
    theta = rng.uniform(0.0, 2.0 * np.pi, n)
    v = speed * np.column_stack([np.cos(theta), np.sin(theta)])
    return v - v.mean(axis=0)


# built from scratch in §1.6; restated here as an instrument.
# It drives whatever force routine it is handed — that is, the `lj_forces`
# built in Exercise 3 — and only records what that routine returns. The
# update rules are the lesson of §1.6; the loop that stores their output is
# bookkeeping, and this notebook's lesson is the layer around it.
def verlet_run(force_fn, r, v, L, dt, n_steps, sample_every=1, **force_kwargs):
    """Integrate eq-md-verlet in a periodic cell, sampling the trajectory.

    Kick-drift-kick velocity Verlet at one force evaluation per step, the
    symplectic stepper of §1.6. Positions are NOT wrapped back into the
    cell: periodicity lives entirely in the minimum-image displacement
    inside the force routine, so an atom is free to walk off to its
    fourth periodic copy and the physics never notices.

    Parameters
    ----------
    force_fn : callable
        Force routine ``(r, L, **force_kwargs) -> (U, F, W)`` giving the
        potential energy, the (n, 2) forces, and the virial sum.
    r, v : numpy.ndarray
        Initial positions and velocities, shape (n, 2); copied, not mutated.
    L : float
        Side of the periodic cell.
    dt : float
        Time step.
    n_steps : int
        Number of steps to take.
    sample_every : int, optional
        Record every k-th step (default 1).
    **force_kwargs
        Passed straight through to ``force_fn`` (this is how Exercise 5
        switches the minimum image and the potential shift off).

    Returns
    -------
    dict
        Keys ``t``, ``U``, ``K``, ``W`` (sampled series), ``R``, ``V``
        (sampled positions and velocities), and ``r``, ``v`` (the final
        state, for chaining one run onto the next).
    """
    r = r.copy()
    v = v.copy()
    U, F, W = force_fn(r, L, **force_kwargs)
    t_s, U_s, K_s, W_s = [0.0], [U], [0.5 * float((v * v).sum())], [W]
    R_s, V_s = [r.copy()], [v.copy()]
    for k in range(1, n_steps + 1):
        v += 0.5 * dt * F  # kick
        r += dt * v  # drift
        U, F, W = force_fn(r, L, **force_kwargs)
        v += 0.5 * dt * F  # kick
        if k % sample_every == 0:
            t_s.append(k * dt)
            U_s.append(U)
            K_s.append(0.5 * float((v * v).sum()))
            W_s.append(W)
            R_s.append(r.copy())
            V_s.append(v.copy())
    return {
        "t": np.array(t_s),
        "U": np.array(U_s),
        "K": np.array(K_s),
        "W": np.array(W_s),
        "R": np.array(R_s),
        "V": np.array(V_s),
        "r": r,
        "v": v,
    }

## Exercise 1 — The Lennard-Jones pair force

The 12-6 potential {eq}`eq-md-lj` has appeared in this course once
before, in [§0.13](../00-foundations/optimization.ipynb), purely as a
scalar function with a minimum for a minimizer to find. Nothing has
ever differentiated it. That step, from a potential to the vector
force {eq}`eq-md-force` that an integrator can use, is the first thing
a molecular-dynamics program does and the first thing to get wrong,
so it gets its own exercise and its own certification.

Two features of the implementation are worth stating before writing
it. The inner loop of every molecular-dynamics code sees the
*squared* separation $r^2$ and never takes a square root: both
$u$ and the combination $f(r)/r$ needed by {eq}`eq-md-force` are
rational functions of $r^{-2}$, so the natural interface takes $r^2$
and returns $u$ together with $f/r$, and the caller multiplies by the
separation vector. And the whole thing must be written as array
arithmetic, because Exercise 3 will hand it an entire $N \times N$
table of squared separations at once.

The certification spends four facts fixed in advance. In reduced
units $u(\sigma) = u(1) = 0$ by the definition of $\sigma$; the
minimum sits at $r_{\min} = 2^{1/6} = 1.122462\ldots$ where
$u = -\varepsilon = -1$ exactly and the force vanishes; and, the sharp
test, the analytic force must agree with a central finite difference
of the analytic potential, $-[u(r+h)-u(r-h)]/2h$, which is the
numerical-differentiation instrument of
[§0.3](../00-foundations/quadrature-differentiation.ipynb) used as a
referee. A sign slip, a dropped factor of two, or a mis-differentiated
exponent survives the first three checks and dies on the fourth.

**Part a)** Write `lj_pair(r2)` taking an array of squared separations
and returning the pair `(u, f_over_r)`: the potential of
{eq}`eq-md-lj` and the scalar $24[2r^{-12} - r^{-6}]/r^{2}$ that
{eq}`eq-md-force` multiplies by $\mathbf r_{ij}$. Build both from
powers of `inv2 = 1.0 / r2` alone, with no `numpy.sqrt`.
**Write this one yourself** — the implementation is the lesson.

**Part b)** Verify the three anchors: `lj_pair` returns $u = 0$ at
$r = 1$ (`atol=1e-12`), and at $r = 2^{1/6}$ returns $u = -1$
(`rtol=1e-12`) with $|f| < 10^{-12}$.

**Part c)** Verify the differentiation: at $r = 1.2$, $1.5$ and
$2.0$, compare the analytic radial force $(f/r)\cdot r$ against the
central difference of $u$ at step $h = 10^{-5}$, and check they agree
to `rtol=1e-6`.

**Part d)** Evaluate the potential at the cutoff, $u(2.5)$, and
confirm it is $-0.0163169$ (`rtol=1e-5`): the small number that
Exercise 5 will show is not small enough to ignore.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([u_sigma]),
    np.array([0.0]),
    "the potential crosses zero at r = sigma, which is what defines sigma",
    atol=1e-12,
)
validate.close(
    np.array([u_min, f_min * R_MIN]),
    np.array([-1.0, 0.0]),
    "and bottoms out at exactly -epsilon at r_min = 2^(1/6), where the "
    "force vanishes: the well of eq-md-lj, in reduced units",
    rtol=1e-12,
    atol=1e-12,
)
validate.close(
    f_analytic,
    f_numeric,
    "the differentiation is right: the analytic force of eq-md-force "
    "matches a central difference of eq-md-lj at r = 1.2, 1.5, 2.0",
    rtol=1e-6,
)
validate.close(
    np.array([u_cut]),
    np.array([-0.0163169]),
    "and the potential still carries -0.0163 epsilon at the cutoff "
    "r_c = 2.5 — under two percent of the well, and not negligible",
    rtol=1e-5,
)

## Exercise 2 — A box with no walls, and the minimum image

A simulation of a few dozen atoms in a box with walls is a simulation
of walls. The repair is to give the cell no boundary at all: tile the
plane with copies of the cell of side $L$, so that an atom leaving one
edge enters at the opposite one and every atom sees a full
neighbourhood. The bulk liquid is then modelled by a system with no
surface.

The subtlety, and the reason this exercise exists, is what
periodicity does to *distances*. Two atoms near opposite edges have
coordinates differing by nearly $L$, and are nevertheless neighbours:
each sits next to a periodic image of the other. Taking coordinate
differences at face value therefore reports the wrong separation for
exactly the pairs that matter, and the fix is {eq}`eq-md-minimage`,
the **minimum-image convention**, which replaces each component of
the separation by its representative in $[-L/2, L/2)$. The phrase does
not appear anywhere else in this course, and it is the single largest
gap between what the reader has built and a working simulation.

The figure below shows the situation in the only way that makes it
obvious: the central cell with its eight nearest copies, one pair of
atoms, and the two candidate separation vectors between them.

In [ ]:
# (solution hidden on the public site)


The implementation is one line of arithmetic per component, and it is
worth writing rather than reading: `numpy.round` rounds half to even,
which is exactly the tie-breaking a separation of precisely $L/2$
needs, and the expression must broadcast over an entire
$(N, N, 2)$ table of separations because that is how Exercise 3 will
call it.

What certifies it is a brute-force search. For a given pair there are
only nine candidate images in two dimensions (the cell itself and its
eight neighbours), so the nearest one can be found by simply trying
all nine and keeping the shortest, at a cost no production code would
pay and with a correctness no arithmetic trick can beat. Agreement
between the two must be *exact*, not approximate: both compute the
same displacement vector, so any difference is a bug rather than a
tolerance.

The convention also imposes a constraint. If the interaction reached
beyond $L/2$, an atom could interact with two images of the same
partner and the "minimum" image would no longer be well defined. So
$r_c \le L/2$ always, and the box used throughout this notebook
($N = 64$ atoms at density $\rho = 0.4$, hence
$L = \sqrt{N/\rho} = 12.649$) satisfies it with room to spare.

**Part a)** Write `min_image(d, L)`, returning the minimum-image form
of a displacement array `d` of any shape, using
`d - L * numpy.round(d / L)` from {eq}`eq-md-minimage`.
**Write this one yourself** — the implementation is the lesson.

**Part b)** Verify it against brute force: for 500 pairs of points
drawn uniformly in a cell of side $L = 10$ with
`numpy.random.default_rng(5170)`, compare `min_image` against an
explicit search over the nine translations
$\mathbf d + L(n_x, n_y)$, $n_x, n_y \in \{-1, 0, 1\}$, keeping the
shortest. Require exact agreement (`atol=0`, `rtol=0`) on every
component.

**Part c)** Verify the two properties the convention guarantees on
that same sample: no component of a minimum-image displacement
exceeds $L/2$ in magnitude, and no minimum-image distance exceeds the
half-diagonal $L/\sqrt2 = 7.071$. Then confirm that the notebook's
own box clears the consistency constraint, $r_c = 2.5 \le L/2 = 6.325$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    d_fast,
    d_brute,
    "the one-line minimum image of eq-md-minimage returns exactly what "
    "an exhaustive search over all nine periodic images returns, for "
    "500 random pairs",
    rtol=0.0,
    atol=0.0,
)
validate.check(
    max_component <= L_TEST / 2 + 1e-12
    and max_distance <= L_TEST / np.sqrt(2.0) + 1e-12,
    "and it honours both guarantees: no component past L/2, no distance "
    "past the half-diagonal L/sqrt(2)",
    f"max component {max_component:.4f}, max distance {max_distance:.4f}",
)
validate.check(
    R_CUT <= L_BOX / 2,
    "the box used from here on clears the consistency constraint "
    "r_c <= L/2, so no atom ever meets two images of the same partner",
    f"r_c = {R_CUT} vs L/2 = {L_BOX / 2:.4f}",
)

## Exercise 3 — The machine: forces, energy, and the virial in one pass

Now the assembly. The pairwise structure is already familiar: the
$O(N^2)$ broadcast force sum of
[§1.8](../01-elementary-mechanics/solar-system.ipynb) built the
$(N, N, 2)$ table of separations and contracted it against the
masses without a Python loop over pairs, and the same skeleton carries
over unchanged. Three things are new, and all three belong to the
layer outside the integrator: the separations are passed through
{eq}`eq-md-minimage` before anything else touches them, pairs beyond
$r_c$ are switched off and the surviving ones shifted by $-u(r_c)$ as
in {eq}`eq-md-shift`, and the routine accumulates the **virial**
$W = \sum_{i<j}\mathbf r_{ij}\cdot\mathbf f_{ij}$ of
{eq}`eq-md-virial` alongside the energy, because it costs nothing here
and cannot be reconstructed later from the trajectory alone.

One numerical detail deserves care rather than a trick. The diagonal
of the separation table is zero, and $1/0$ is where the naive version
dies. Rather than masking after the fact, set the diagonal of the
*squared* separations to any value comfortably beyond the cutoff (for
instance $4r_c^2$): the self-pair then falls outside the interaction
range by construction, contributes exactly zero to $U$, $F$ and $W$,
and no division by zero ever happens.

The certifications are the sharpest in the notebook, because both
quantities can be obtained a second way from the energy alone. The
force must be $-\nabla U$, so displacing one atom by $\pm h$ and
central-differencing the *total* potential energy must reproduce that
atom's force. And the virial is a derivative too: scaling every
coordinate and the box together, $\mathbf r \to \lambda\mathbf r$,
$L \to \lambda L$, gives
$\mathrm{d}U/\mathrm{d}\lambda|_{\lambda=1} = \sum_{i<j} u'(r_{ij})r_{ij} = -W$,
so a central difference in $\lambda$ measures the virial without ever
forming a force. That second check is a strong one: it is sensitive to
the factor $\tfrac12$ on the double sum, to the sign, and to the
cutoff bookkeeping, and it only works cleanly *because* the potential
was shifted, since a discontinuous $u$ would make $U(\lambda)$
discontinuous too.

The test configuration for all three checks is the $8\times8$ square
lattice at $\rho = 0.4$ (so $L = 12.649$) displaced by Gaussian noise
of standard deviation $0.05\,\sigma$ from `numpy.random.default_rng(5173)`:
a disordered configuration with a spread of separations, so the checks
probe the repulsive core and the attractive tail at once.

**Part a)** Write `lj_forces(r, L, r_cut=R_CUT, shift=True, minimum_image=True)`
returning `(U, F, W)`. Build the $(N, N, 2)$ separation table
`r[:, None, :] - r[None, :, :]`, pass it through your `min_image` when
`minimum_image` is true, square-sum to $r^2$, set the diagonal beyond
the cutoff with `numpy.fill_diagonal`, call your `lj_pair`, zero
everything outside $r_c$ with `numpy.where`, subtract $u(r_c)$ from the
surviving energies when `shift` is true, and return
$U = \tfrac12\sum_{i\ne j}u$, $F_i = \sum_j (f/r)_{ij}\mathbf r_{ij}$,
and $W = \tfrac12\sum_{i \ne j}(f/r)_{ij}\,r_{ij}^2$. The two boolean
flags exist so that Exercise 5 can switch each convention off and
watch what happens. The virial accumulator in particular has no
second source: it cannot be reconstructed from a saved trajectory,
so if it is not written here it is not available at all.
**Write this one yourself** — the implementation is the lesson.

**Part b)** Verify $F = -\nabla U$: displace atom $0$ by
$\pm h = \pm 10^{-6}$ in $x$ and in $y$, central-difference the total
potential energy, and check both components against `lj_forces` to
`rtol=1e-6`.

**Part c)** Verify the virial: evaluate $U$ at
$\lambda = 1 \pm 10^{-6}$ with all coordinates *and* the box scaled by
$\lambda$, and check $-\mathrm{d}U/\mathrm{d}\lambda$ against the
returned $W$ to `rtol=1e-6`.

**Part d)** Verify Newton's third law: the total force
$\sum_i \mathbf F_i$ vanishes to $10^{-12}$, which is what will keep
the total momentum pinned at zero for the whole of the rest of this
notebook.

```{admonition} With your assistant
:class: tip
The $O(N^2)$ table above is honest and, past a few hundred atoms,
hopeless: almost every entry it computes is a pair beyond the cutoff.
Ask your assistant for a **cell-list** version of `lj_forces` that bins
the atoms into square cells of side $\ge r_c$ and visits only the
nine neighbouring bins of each cell. Then check it the way this
exercise checked the original: on the same jostled lattice, its
forces must agree with the $O(N^2)$ routine to `rtol=1e-12`
elementwise, and its $W$ must still equal $-\mathrm{d}U/\mathrm{d}\lambda$.
A cell list that quietly drops the periodic wrap of the bin indices
passes neither. The check is yours.
```

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    F_test[0],
    F0_numeric,
    "the force routine really returns -grad U: both components on atom "
    "0 match a central difference of the total potential energy",
    rtol=1e-6,
)
validate.close(
    np.array([W_test]),
    np.array([W_numeric]),
    "and the virial accumulator is right: W equals -dU/dlambda under a "
    "uniform scaling of coordinates and box together, factor of one "
    "half and all",
    rtol=1e-6,
)
validate.check(
    net_force < 1e-12,
    "Newton's third law holds pairwise, so the total force vanishes — "
    "which is what pins the total momentum for the rest of the notebook",
    f"|sum F| = {net_force:.1e}",
)

## Exercise 4 — Temperature, equilibration, and production

With a force routine that has been certified twice against the energy,
the simulation can be switched on. What is still missing is a
thermometer and a protocol.

The thermometer is equipartition, {eq}`eq-md-temp`, with the
finite-size correction that textbooks state for the three-dimensional
case and that introductory treatments usually drop: the total momentum
was set to zero at the start and Newton's third law (Exercise 3, Part
d) holds it there, so two of the $2N$ velocity components are
constrained and the divisor is $2N-2$, not $2N$. At $N = 64$ that is a
$1.6\%$ shift in every temperature this notebook reports. Two things
{eq}`eq-md-temp` is *not*: it is not an instantaneous quantity (the
per-sample $2K/(2N-2)$ swings by tens of percent, and only its time
average in equilibrium deserves the name temperature), and it is not a
canonical temperature, since the run has a fixed energy rather than a
fixed temperature.

The protocol is two phases, kept strictly apart. **Equilibration**
starts from an artificial configuration (here the $8\times8$ square
lattice, which is not even the right crystal structure for a
two-dimensional Lennard-Jones solid) and destroys it, rescaling the
velocities by $\sqrt{T_{\rm target}/T}$ every hundred steps to steer
the run toward $T = 1.0$. Nothing measured here is reported. Then the
rescaling stops and **production** runs pure NVE: constant energy,
no interference, and every number in the notebook comes from this
phase. The discipline of discarding the transient before averaging is
the one [§5.8](partition-function.ipynb) already taught for Metropolis
chains, and it transfers unchanged.

How does one know the transient is over? Not by watching the energy,
which is conserved from the first step and says nothing about
equilibrium. The usable signal is **stationarity of the slow
observables**: the potential energy per atom falls steeply while the
lattice melts and then stops falling, and successive windows of the
production run give the same average to within their scatter. That is
a necessary condition, not a sufficient one, and Exercise 8 turns
"within their scatter" into a number. Here the crude version is
enough: the drop across equilibration is an order of magnitude larger
than the spread across production quarters.

The run: $N = 64$ atoms at $\rho = 0.4$ (so $L = 12.649$),
$\Delta t = 0.004$, $6000$ equilibration steps with a rescale every
$100$, then $20{,}000$ production steps sampled every $5$. Initial
velocities are the Setup's `shell_velocities` at speed
$\sqrt{2T_{\rm target}}$ from `numpy.random.default_rng(517)`, which is
also the non-Maxwellian specimen Exercise 7 studies.

**Part a)** Write `temperature(v)`, the instantaneous kinetic
temperature of {eq}`eq-md-temp`: the sum of $|\mathbf v_i|^2$ divided
by $2N-2$ in these units.

**Part b)** Equilibrate: from the lattice and the shell velocities,
take sixty blocks of $100$ steps with the Setup's `verlet_run`,
rescaling `v` by $\sqrt{T_{\rm target}/\texttt{temperature(v)}}$ after
each block. Record the potential energy per atom throughout, and
verify it falls by more than $0.4\,\varepsilon$ per atom from the
lattice value.

**Part c)** Produce: run $20{,}000$ NVE steps from the equilibrated
state, sampling every $5$. Verify that the total energy $K + U$ is
conserved, $\max|E(t) - E(0)| / (Nk_BT) < 2\times10^{-3}$, and that
the total momentum $|\sum_i \mathbf v_i|$ stays below $10^{-10}$ for
the whole run, which is the premise the $2N-2$ of {eq}`eq-md-temp`
rests on.

**Part d)** Verify stationarity: split the production series into four
equal quarters and check that the quarter means of $U/N$ span less
than $0.05\,\varepsilon$, an order of magnitude below the
equilibration drop measured in Part b. Report the production
temperature both ways, with $2N-2$ and with the naive $2N$, and
confirm the two differ by the expected factor $N/(N-1)$ to
`rtol=1e-12`: a $1.6\%$ bookkeeping choice that no amount of sampling
will average away.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    equil_drop > 0.4,
    "equilibration does real work: the potential energy per atom falls "
    "by more than 0.4 epsilon as the square lattice is destroyed",
    f"drop {equil_drop:.4f} epsilon per atom",
)
validate.check(
    energy_drift < 2e-3,
    "and the production run conserves energy, as a symplectic stepper "
    "on a continuous potential must: max|E - E0| well under a "
    "thousandth of N k_B T",
    f"max|E - E0| / (N k_B T) = {energy_drift:.2e}",
)
validate.check(
    momentum < 1e-10,
    "the total momentum stays pinned at zero for the whole run, which "
    "is what makes the 2N-2 of eq-md-temp the honest divisor",
    f"max |sum of velocities| = {momentum:.1e}",
)
validate.check(
    quarter_spread < 0.05 and quarter_spread < 0.2 * equil_drop,
    "the four production quarters agree on U/N to well inside the "
    "equilibration drop: stationary, so the transient is behind us",
    f"quarter spread {quarter_spread:.4f} vs drop {equil_drop:.4f}",
)
validate.close(
    np.array([T_prod / T_naive]),
    np.array([N_ATOMS / (N_ATOMS - 1)]),
    "and the finite-size correction is exactly the factor N/(N-1): a "
    "1.6 percent choice at N = 64 that no amount of sampling removes",
    rtol=1e-12,
)

The periodic cell is easiest to believe once it is watched. The
animation below replays the production trajectory with the positions
wrapped back into the cell for display only (the integrator never
wraps them: periodicity lives entirely inside {eq}`eq-md-minimage`).
One atom is tracked in amber, and the thing to watch for is the moment
it leaves an edge and reappears at the opposite one without anything
happening to the physics.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
unwrapped_span = float(np.ptp(prod["R"]))
validate.check(
    float(pos_wrapped.min()) >= 0.0 and float(pos_wrapped.max()) < L_BOX,
    "every displayed position lies inside the cell, and the unwrapped "
    "trajectory behind them travelled further than the cell is wide: "
    "the atoms really are crossing the boundary",
    f"unwrapped span {unwrapped_span:.1f} sigma vs L = {L_BOX:.2f}",
)
validate.check(
    unwrapped_span > L_BOX,
    "and the animation is not a still life: the unwrapped coordinates "
    "range over more than one box length during the run",
    f"unwrapped coordinate range {unwrapped_span:.2f} sigma",
)

## Exercise 5 — What the two boundary decisions cost

Both conventions of the last two exercises look like housekeeping, and
both are load-bearing. This exercise removes each in turn from the
*same* equilibrated starting state and measures the damage, which is
the only way to know how much they were doing.

**Forgetting the minimum image** is not an exotic mistake. The
tempting thing is to wrap every atom back into the cell with
`r % L` (which looks like periodicity) and then take plain coordinate
differences (which is not). Two atoms straddling an edge are then
reported at separation $\approx L$ instead of $\approx 0$, so their
interaction silently vanishes; worse, the instant an atom crosses the
boundary its whole set of separations jumps discontinuously, and with
them the force and the potential energy. The result is not a small
error. Energy is injected at every crossing, the system heats itself,
atoms are driven into each other's repulsive cores, and the run
destroys itself within a few reduced time units.

**Forgetting the shift** is subtler and, for that reason, more
dangerous. Truncating {eq}`eq-md-lj` at $r_c$ without subtracting
$u(r_c) = -0.0163$ changes no force at all, so the trajectory is
*bit-for-bit identical*. What changes is that $K + U$ is no longer a
constant of the motion: every pair that drifts across $r_c$ steps the
total energy by $u(r_c)$, and the accumulated random walk of those
steps looks exactly like integrator drift. A simulator who monitors
energy conservation to certify the time step, as
[§1.6](../01-elementary-mechanics/integrators.ipynb) taught, will
shorten the step, see no improvement, and blame the wrong thing. The
deeper statement is that the unshifted truncation is not a Hamiltonian
system: a discontinuous potential has no conserved energy to find.

Both runs start from the state produced at the end of Exercise 4 and
last $4000$ steps at $\Delta t = 0.004$. The measure throughout is
$\max|E(t) - E(0)|/(Nk_BT)$, which is scale-free and comparable
between runs, unlike a relative drift on a total energy that happens
to be a small difference of two large numbers.

**Part a)** Run the correct machine for $4000$ steps as the reference
and record its energy excursion.

**Part b)** Write the naive alternative: a force routine that wraps
positions with `r % L` and calls your `lj_forces` with
`minimum_image=False`. Run it for $4000$ steps from the same state and
verify the failure is catastrophic rather than gradual: its energy
excursion exceeds $Nk_BT$ within the first reduced time unit, while
the reference stays below $10^{-3}Nk_BT$ over the same window.

**Part c)** Run `lj_forces` with `shift=False` for $4000$ steps from
the same state. Verify two things: the sampled positions are
*identical* to Part a to `atol=0` (shifting a potential by a constant
cannot move an atom), and the energy excursion is more than ten times
the reference. Confirm the size is what {eq}`eq-md-shift` predicts by
checking that the excursion, converted back to units of $u(r_c)$,
corresponds to a few tens of pair crossings.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    exc_nomi[early].max() > 1.0 and exc_ok[early].max() < 1e-3,
    "forgetting the minimum image is not a small error: the energy "
    "leaves N k_B T within one reduced time unit, while the correct "
    "run has not moved a thousandth of it",
    f"{exc_nomi[early].max():.2e} vs {exc_ok[early].max():.2e} at t <= 1",
)
validate.close(
    run_ok["R"],
    run_noshift["R"],
    "shifting the potential by a constant moves no atom: the shifted "
    "and unshifted runs trace bit-for-bit the same trajectory",
    rtol=0.0,
    atol=0.0,
)
validate.close(
    jump_unshifted,
    abs(u_cut),
    "and yet a single pair walked across the cutoff makes the unshifted energy "
    "JUMP, by exactly the u(r_c) that was never subtracted",
    rtol=1e-6,
)
validate.close(
    jump_unshifted_fine,
    abs(u_cut),
    "and it is a true discontinuity, not a steep slope: shrinking the step "
    "tenfold leaves the jump exactly where it was",
    rtol=1e-6,
)
validate.check(
    abs(jump_shifted) < 1e-8 and 8.0 < shifted_ratio < 12.0,
    "while the shifted potential carries the same pair across continuously — "
    "its step is proportional to how far you stepped, and vanishes with it",
    f"shifted step {abs(jump_shifted):.2e} falls to {abs(jump_shifted_fine):.2e} "
    f"when eps drops tenfold (ratio {shifted_ratio:.1f}), against an unshifted "
    f"jump of {jump_unshifted:.6f} that does not move",
)
validate.check(
    exc_noshift.max() > 3.0 * exc_ok.max(),
    "and over a long run those jumps accumulate: the energy stops standing "
    "still, with no integrator to blame",
    f"{exc_noshift.max():.2e} vs {exc_ok.max():.2e} "
    f"({exc_noshift.max() / exc_ok.max():.1f}x; the exact factor is not "
    "reproducible across machines, since the trajectory is chaotic and the "
    "number of crossings in a fixed window is not)",
)
validate.check(
    5.0 < crossings < 500.0,
    "the size is exactly the arithmetic of eq-md-shift: the excursion "
    "is a few tens of steps of u(r_c), one per pair crossing the cutoff",
    f"{crossings:.0f} units of |u(r_c)| = {abs(u_cut):.4f}",
)

## Exercise 6 — Why velocity rescaling is not a thermostat

Equilibration above used velocity rescaling, and it worked: the run
arrived where it was steered. It is worth being explicit about what
that manoeuvre is and is not, because the introductory literature
almost universally sidesteps the question by running NVE and saying
nothing, while the graduate literature
{cite}`allen_tildesley,frenkel_smit` states it plainly. This notebook
states it plainly.

A canonical ensemble at temperature $T$ does not hold the kinetic
energy fixed. It lets it fluctuate, and the size of the fluctuation is
not free: $K$ is a sum of $d_f = 2N-2$ independent squared Gaussian
velocity components, so it follows a gamma distribution with
$\langle K\rangle = \tfrac{d_f}{2}k_BT$ and
$\mathrm{Var}(K) = \tfrac{d_f}{2}(k_BT)^2$, giving the relative width

```{math}
:label: eq-md-kfluct
\frac{\sigma_K}{\langle K\rangle} \;=\; \sqrt{\frac{2}{d_f}}
  \;=\; \sqrt{\frac{2}{2N-2}} \;=\; 0.1260 \quad (N = 64),
```

the same $1/\sqrt N$ narrowing that
[§5.9](grand-canonical-ensemble-equivalence.ipynb) used to prove the
ensembles equivalent. A scheme that rescales every velocity to hit a
target temperature at every step produces $\sigma_K = 0$ exactly. It
therefore samples a distribution with the right mean and the wrong
fluctuations, which is to say the wrong ensemble; the Berendsen
scheme, which rescales gently rather than exactly, is the same
objection softened but not answered. Getting the fluctuations right
takes a genuine thermostat: Nosé-Hoover, canonical velocity
rescaling, or the Langevin dynamics whose BAOAB integrator
[§5.11](taste-of-nonequilibrium.ipynb) already built for
non-interacting particles.

NVE is not canonical either, and this exercise is honest about that
too. At fixed total energy the kinetic energy still fluctuates, since
it trades with the potential energy, but by less than
{eq}`eq-md-kfluct` demands, because the exchange is with a finite
rather than an infinite reservoir. The point is not that NVE is the
canonical ensemble. The point is that NVE is a *well-defined* ensemble
with a name, whereas per-step rescaling is not.

**Part a)** Run $2000$ steps from the equilibrated state of Exercise 4,
rescaling `v` by $\sqrt{T_{\rm target}/\texttt{temperature(v)}}$ after
*every* step, and record the kinetic energy at each step.

**Part b)** Compare three numbers: the relative width
$\sigma_K/\langle K\rangle$ of that rescaled run, of the Exercise 4
production run, and the canonical requirement $\sqrt{2/(2N-2)} = 0.1260$
of {eq}`eq-md-kfluct`. Verify the rescaled run's width is below
$10^{-10}$, so it misses the canonical value by more than nine orders
of magnitude, and that the NVE width lies between $30\%$ and $90\%$ of
the canonical one: smaller, as a finite reservoir requires, but not
zero.

**Part c)** Verify the other half of the objection. The equilibration
ended with an *exact* rescale to $T = 1.0$, yet the NVE run that
followed averages $\langle T\rangle = 0.947$: take the mean of the
Exercise 4 temperature series with `numpy.ndarray.mean` and check
that it differs from the target by more than $2\%$. Setting
the instantaneous kinetic energy is not the same as setting the
temperature of the state that follows, because the potential energy
takes its own share the moment the run is released.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    width_rescaled < 1e-10,
    "per-step velocity rescaling holds the kinetic energy fixed to "
    "machine precision, so it produces none of the fluctuation "
    "eq-md-kfluct requires: it is not the canonical ensemble",
    f"sigma_K/<K> = {width_rescaled:.1e} against a required " f"{width_canonical:.4f}",
)
validate.check(
    0.30 < width_nve / width_canonical < 0.90,
    "and NVE is not canonical either, honestly: its kinetic energy "
    "does fluctuate, but by less, because the reservoir it trades with "
    "is finite",
    f"NVE width {width_nve:.4f} = {width_nve / width_canonical:.2f} " f"of canonical",
)
validate.check(
    T_offset > 0.02,
    "setting the instantaneous kinetic energy is not setting the "
    "temperature: rescaled to exactly 1.000 and released, the run "
    "settles a full 5 percent below it",
    f"<T> = {T_prod:.4f} against a target of {T_TARGET:.3f}",
)

## Exercise 7 — Maxwell and Boltzmann out of pure Newtonian flow

The Maxwell-Boltzmann distribution has appeared twice already in this
volume, and neither time did a smooth force law produce it.
[§5.11](taste-of-nonequilibrium.ipynb) reached it by applying
*random* energy- and momentum-conserving collisions to randomly chosen
pairs, which is molecular chaos assumed rather than derived, and
[§5.12](kinetic-theory.ipynb) by hard-core impulses in an event-driven
gas. This exercise closes the gap. The dynamics below is deterministic,
time-reversible, generated by a smooth potential, and contains no
randomness whatever after the initial condition is drawn. Nothing in it
knows about Boltzmann. Yet a velocity distribution that could hardly be
less Maxwellian becomes Maxwellian, in a handful of collision times.

The initial condition is the Setup's `shell_velocities`: every atom
given the *same* speed with a random direction, so the speed
distribution starts as a delta function and the velocity components
start with the arcsine distribution of $v_0\cos\theta$, which is
bimodal and about as far from a Gaussian as a symmetric distribution
gets. Positions are taken from an already-equilibrated configuration,
so the potential energy starts where it belongs and the only thing out
of equilibrium is the velocity distribution itself.

In two dimensions the target is precise. Each Cartesian component
should become Gaussian with variance $k_BT$, so the *speed* follows
the Rayleigh distribution

```{math}
:label: eq-md-rayleigh
p(v) \;=\; \frac{v}{k_BT}\,e^{-v^{2}/2k_BT},
\qquad \langle v^2\rangle = 2k_BT,
\qquad \langle v^4\rangle = 8(k_BT)^2 .
```

Two consequences of {eq}`eq-md-rayleigh` are worth more than any
histogram, because they contain **no free parameter at all**: the
dimensionless ratio $\langle v^4\rangle/\langle v^2\rangle^2$ must be
exactly $2$, and the relative width of the speed distribution,
$\sigma_v/\langle v\rangle$, must be $\sqrt{4/\pi - 1} = 0.5227$.
Both are $1$ and $0$ respectively for the initial shell. Neither can
be fitted into agreement, which is what makes them a test rather than
a demonstration.

The third test is a genuine goodness-of-fit statistic rather than an
eyeball: a Kolmogorov-Smirnov test (`scipy.stats.kstest`) of the
pooled velocity components against a standard normal. One piece of
care is needed, and it is stated rather than hidden. The components
must be standardized by *some* temperature, and using the temperature
of the same sample would make the test circular; so the scale is taken
from the second quarter of the run and the test applied to snapshots
from the second *half*, which are a disjoint set of samples. Snapshots
are pooled at a spacing of $2$ reduced time units, several velocity
correlation times apart, so that pooling does not badly overstate the
sample size.

The system: $N = 144$ atoms at $\rho = 0.4$ (so $L = 18.97$),
equilibrated for $4000$ steps with rescaling toward $T = 1$ from
`numpy.random.default_rng(5172)`, then given the shell velocities at
the temperature it had reached, then run $6000$ NVE steps with a
snapshot every $25$.

**Part a)** Build and equilibrate the $12\times12$ system, then replace
its velocities with `shell_velocities` rescaled to the temperature the
equilibrated state had, so the total energy is unchanged and only the
*shape* of the distribution is out of equilibrium.

**Part b)** Run $6000$ NVE steps and compute, at $t = 0$ and averaged
over the second half of the run, the two parameter-free shape
statistics $\langle v^4\rangle/\langle v^2\rangle^2$ and
$\sigma_v/\langle v\rangle$. Verify that at $t = 0$ they sit at
$1$ and $0$ (`atol=0.05` on both, since removing the net momentum
perturbs the common speed slightly), and that averaged over the
second half of the run they have become the Rayleigh values $2$ and
$\sqrt{4/\pi-1} = 0.5227$ (`rtol=5e-2`).

**Part c)** Apply `scipy.stats.kstest` to the velocity components,
standardized as described above. Verify the late-time KS statistic
falls below the $5\%$ critical value $1.36/\sqrt{n}$ for the pooled
sample, and that the initial statistic is more than three times
larger: the shell is rejected as Gaussian, the relaxed gas is not.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([kurt_0, width_0]),
    np.array([1.0, 0.0]),
    "the run starts emphatically non-Maxwellian: every atom at one "
    "speed, so <v^4>/<v^2>^2 = 1 and the speed spread is zero",
    atol=0.05,
)
validate.close(
    np.array([kurt_late, width_late]),
    np.array([2.0, RAYLEIGH_WIDTH]),
    "and smooth deterministic Newtonian flow delivers Maxwell-Boltzmann: "
    "both parameter-free shape statistics land on their Rayleigh values",
    rtol=5e-2,
)
validate.check(
    ks_late.statistic < d_critical,
    "the Kolmogorov-Smirnov test agrees, on velocity components "
    "standardized by a temperature measured in a DISJOINT window: the "
    "relaxed gas is not distinguishable from Gaussian at 5 percent",
    f"D = {ks_late.statistic:.4f} against a critical {d_critical:.4f}",
)
validate.check(
    ks_start.statistic > 3.0 * ks_late.statistic and ks_start.pvalue < 0.01,
    "while the initial shell is rejected outright — the test has teeth, "
    "and it is the dynamics that changed the answer",
    f"D = {ks_start.statistic:.4f} (p = {ks_start.pvalue:.1e}) at t = 0",
)

## Exercise 8 — Error bars on a correlated trajectory

Every number this notebook has reported so far is a mean over a time
series, and not one of them has carried an uncertainty. That is the
next thing to fix, and it is worth saying that the reader will not
find it in the obvious places. Across the standard computational-physics
texts that teach molecular dynamics — Gould, Tobochnik and Christian
{cite}`gould_tobochnik` among them — the molecular-dynamics chapter
teaches the integrator, the potential and the boundary conditions, and
not the error analysis; the research monographs do treat it, but exile
it to an appendix or a late chapter of its own
{cite}`frenkel_smit,allen_tildesley`. The reader is left to infer that
quoting $\langle T\rangle = 0.947$ with no bar is normal practice. It
is normal practice, and it is wrong. Ambegaokar and Troyer
{cite}`ambegaokar2010` is the pedagogical treatment to read.

The problem is correlation. [§5.8](partition-function.ipynb) already
taught half the discipline: discard the transient before averaging,
which Exercise 4 did. What was never priced is what remains. The
temperature two steps from now is very nearly the temperature now, so
the $n = 4001$ samples of the production run are nothing like $4001$
independent measurements, and the familiar $\sigma/\sqrt{n}$ is far too
small. The correct statement is {eq}`eq-md-tau`: the variance of the
mean is inflated by $2\tau_{\rm int}$, so the honest bar is
$\sqrt{2\tau_{\rm int}}$ times the naive one and the run is worth
$n_{\rm eff} = n/2\tau_{\rm int}$ independent samples. The same
arithmetic governs the Metropolis chains of
[§5.10](ising-emergence-universality.ipynb), and
[§0.11](../00-foundations/random-numbers-monte-carlo.ipynb) warned
from the start that Markov-chain samples are bought at the price of
being correlated.

Estimating $\tau_{\rm int}$ needs one piece of care. The sum in
{eq}`eq-md-tau` cannot run to $n$: the tail of $\rho(t)$ is pure noise,
and including it adds variance without adding signal. Sokal's
self-consistent window takes the smallest $W$ satisfying
$W \ge c\,\tau_{\rm int}(W)$, and $c = 8$ is the standing choice in
this course (a shorter window clips slow tails and shrinks the bars,
which is the failure that manufactures phantom agreement).

**Blocking** is the second, independent route to the same number, and
the more convincing one because it needs no window at all. Cut the
series into blocks of length $\ell$, take the mean of each block, and
compute the standard error of those block means. For $\ell \ll \tau$
the blocks are correlated and the estimate is too small; as $\ell$
grows past $\tau$ the blocks decouple and the estimate rises to a
**plateau**. The plateau is the honest error bar, and watching the
estimate climb to it is the most direct demonstration available that
the naive bar was lying.

**Part a)** Write `tau_int(series, c=8.0)` returning
`(tau, W)`: form the normalized autocovariance $\rho(t)$ with
`numpy.correlate(x, x, mode="full")` on the mean-subtracted series
(keeping the non-negative lags and dividing by $\rho(0)$), accumulate
$\tau(W) = \tfrac12 + \sum_{t=1}^{W}\rho(t)$ with `numpy.cumsum` over
the first half of the lags, and return $\tau$ at the smallest $W$ with
$W \ge c\,\tau(W)$. **Write this one yourself** — the implementation is
the lesson.

**Part b)** Write `blocked_error(series, tau)` returning
`(err, n_blocks)`: split the series into blocks of length
$\lceil 16\tau\rceil$ with `numpy.array_split` and return
$\mathrm{std}(\text{block means}, \mathrm{ddof}=1)/\sqrt{n_{\rm blocks}}$.

**Part c)** Apply both to the temperature, the virial pressure
$P = (K + W/2)/L^2$ from {eq}`eq-md-virial`, and $U/N$ of the Exercise
4 production run. Verify for the temperature series that the blocked
bar exceeds the naive $\sigma/\sqrt{n}$ by the factor
$\sqrt{2\tau_{\rm int}}$ that {eq}`eq-md-tau` predicts, to
`rtol=0.15` — two independent estimators of the same inflation, and a
real test of both.

**Part d)** Demonstrate the plateau: repeat the `numpy.array_split`
blocking of Part b at block lengths $1, 2, 4, 8, 16, 32$ times
$\tau_{\rm int}$, and verify the estimate rises monotonically from
$1\tau$ to $16\tau$ and that the $16\tau$ bar is more than $1.5$
times the $1\tau$ one.

**Part e)** Report the damage: verify that the $4001$-sample
temperature series is worth fewer than $400$ effective independent
samples, and check the window is not an artefact by confirming
$\tau_{\rm int}$ moves by less than $20\%$ between $c = 6$, $8$ and
$10$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([inflation]),
    np.array([np.sqrt(2.0 * tau_T)]),
    "blocking and the correlation time agree on how badly the naive "
    "bar understates: the inflation factor is sqrt(2 tau_int), as "
    "eq-md-tau says",
    rtol=0.15,
)
validate.check(
    np.all(np.diff(creep[:5]) > 0.0) and creep[4] > 1.5 * creep[0],
    "and the blocking curve does what it must: rising monotonically "
    "from one tau to sixteen, then flattening — the plateau is the "
    "honest bar",
    f"{creep[0]:.2e} at 1 tau -> {creep[4]:.2e} at 16 tau "
    f"({creep[4] / creep[0]:.2f} times)",
)
validate.check(
    n_eff < 400.0,
    "so 4001 samples of temperature are worth fewer than 400 "
    "independent measurements: correlation, priced",
    f"n_eff = {n_eff:.0f} from n = {T_series.size}, tau = {tau_T:.1f}",
)
validate.check(
    window_spread < 0.20,
    "and the window is not an artefact: tau_int moves by under 20 "
    "percent across c = 6, 8, 10",
    f"tau = " + ", ".join(f"{t:.1f}" for t in taus_c),
)

## Exercise 9 — Pressure from the virial, and the ideal gas left behind

This volume derived a complete equation of state in
[§5.15](van-der-waals.ipynb) and never measured a pressure. The
machinery to do it has been sitting in `lj_forces` since Exercise 3,
because the virial $W = \sum_{i<j}\mathbf r_{ij}\cdot\mathbf f_{ij}$
was accumulated there and certified against
$-\mathrm{d}U/\mathrm{d}\lambda$. With no walls to push on, the virial
is the *only* route: {eq}`eq-md-virial` in two dimensions reads
$PL^2 = K + W/2$, with $K$ the kinetic energy.

The reference to compare against is the ideal-gas law $P = \rho k_BT$,
and the deviation from it is exactly the physics
[§5.15](van-der-waals.ipynb) modelled: attraction pulls the pressure
below ideal, the excluded volume of the repulsive core pushes it
above, and the van der Waals equation is the crudest interpolation
between those two effects. Here they are measured rather than
modelled. At low density the departure is governed by the second
virial coefficient {eq}`eq-md-b2`, which is a one-dimensional
quadrature over the *same* truncated and shifted potential the
simulation uses, so `scipy.integrate.quad` supplies an independent
prediction to machine precision.

Two pieces of honesty come with the comparison. The first is the
finite-size bookkeeping of {eq}`eq-md-temp` reappearing: since
$2K/2 = (N-1)k_BT$ exactly, the compressibility factor
$Z = PL^2/Nk_BT$ tends to $(N-1)/N = 0.984$ rather than $1$ as
$\rho \to 0$, a $1.6\%$ offset that is arithmetic rather than physics
and is carried in every prediction below. The second is that
{eq}`eq-md-b2` is only the first term of a series: by $\rho = 0.1$ the
third virial coefficient is already visible, and the exercise reports
how much of the departure $B_2$ accounts for rather than pretending it
accounts for all of it.

The protocol per density: $N = 64$ atoms, box $L = \sqrt{N/\rho}$,
equilibration of $4000$ steps with rescaling toward $T = 1$ every
$100$, then three temperature-matching passes (run $3000$ steps, then
rescale by $\sqrt{T_{\rm target}/\langle T\rangle}$ using the *average*
temperature of that probe rather than an instantaneous value, which is
what keeps the eight state points at a common temperature), then
production sampled every $5$ steps: $20{,}000$ steps at
$\rho \le 0.05$ where the virial is a rare-event quantity, $12{,}000$
elsewhere. Seeds are `numpy.random.default_rng(5170 + k)` for the
$k$-th density.

**Part a)** Write `b2_2d(T)`, the second virial coefficient
{eq}`eq-md-b2` of the truncated and shifted potential, by
`scipy.integrate.quad` of $[e^{-u_{\rm ts}(r)/T} - 1]\,r$ from $0$ to
$r_c$ (the integrand vanishes identically beyond $r_c$), times $-\pi$.
Evaluate it at $T = 1$ and report the value.

**Part b)** Run the sweep over
$\rho \in \{0.02, 0.05, 0.10, 0.20, 0.35, 0.50, 0.65, 0.80\}$,
recording for each the mean temperature, the mean pressure
$P = (K + W/2)/L^2$, the compressibility factor
$Z = P/(\rho k_BT)$, and its blocked error bar from your Exercise 8
`tau_int` and `blocked_error`.

**Part c)** Verify the ideal-gas limit: at $\rho = 0.02$ the measured
pressure is within $6\%$ of $\rho k_BT$ (and the $1.6\%$ of that which
is the $(N-1)/N$ offset is bookkeeping, not physics).

**Part d)** Verify the departure grows and changes sign: $Z$ lies
*below* $(N-1)/N$ at every density up to $\rho = 0.20$ (attraction
winning, the van der Waals $a$) and *above* $1$ from $\rho = 0.50$
upward (the core winning, the van der Waals $b$), reaching more than
three times the ideal-gas pressure at $\rho = 0.80$.

**Part e)** Verify the second virial coefficient is doing the work at
low density: at $\rho = 0.10$, the measured departure
$Z - (N-1)/N$ divided by the predicted $B_2(T)\rho$ lies between
$0.5$ and $1.0$, the shortfall being the third virial coefficient that
{eq}`eq-md-b2` does not contain.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([ideal_ratio[0]]),
    np.array([1.0]),
    "at the lowest density the measured virial pressure is the "
    "ideal-gas law, to within a few percent of which 1.6 is the "
    "(N-1)/N bookkeeping of eq-md-temp",
    rtol=6e-2,
)
low, high = rho_s <= 0.20, rho_s >= 0.50
below_by = float(np.max(Z_s[low] + 2.0 * dZ_s[low]) - FINITE_N)
above_by = float(np.min(Z_s[high] - 2.0 * dZ_s[high]) - 1.0)
validate.check(
    below_by < 0.0 and above_by > 0.0,
    "and the departure changes sign exactly as van der Waals says it "
    "must: attraction holds P below ideal at low density, the "
    "repulsive core drives it above at high",
    f"Z = {Z_s[0]:.3f} at rho = 0.02, {Z_s[-1]:.3f} at rho = 0.80; "
    f"the low-density points sit {-below_by:.3f} below the ideal line and the "
    f"high-density ones {above_by:.3f} above it, both after two error bars",
)
# Monotonicity is a claim about a *trend*, and these are measured points with error
# bars, so it must be asserted against those error bars and not through them: two
# state points closer together than their own uncertainties have no order to test.
# (An earlier version demanded a strictly increasing sequence, and went red when two
# adjacent points swapped on a different machine — the trajectory is chaotic, so the
# noise is not reproducible even though the trend is.)
core = rho_s >= 0.20
d_core = np.diff(Z_s[core])
sig_core = np.hypot(dZ_s[core][1:], dZ_s[core][:-1])
no_real_dip = float(np.min(d_core + 2.0 * sig_core))
total_rise = float(Z_s[core][-1] - Z_s[core][0])
rise_sigma = float(total_rise / np.hypot(dZ_s[core][-1], dZ_s[core][0]))
validate.check(
    Z_s[-1] > 3.0 and no_real_dip > 0.0 and rise_sigma > 5.0,
    "the growth is monotone once the core dominates, and the densest "
    "state point carries more than three times the ideal-gas pressure",
    f"Z rises through {', '.join(f'{z:.2f}' for z in Z_s[core])}; "
    f"no decrease survives its error bars (worst {no_real_dip:+.3f}) and the "
    f"total rise is {rise_sigma:.0f} sigma",
)
# The share is a ratio of two small differences, so its error bar is not small:
# gate what the measurement can actually establish, which is that the departure is
# real, has the sign B_2 predicts, and does not exceed what B_2 alone would give.
# Claiming a precise fraction here would be claiming precision we do not have.
share_sigma = float(dZ_s[idx_10] / abs(b2_predicted[idx_10] * rho_s[idx_10]))
validate.check(
    b2_share - 2.0 * share_sigma > 0.0 and b2_share + 2.0 * share_sigma < 1.4,
    "and quadrature judges the simulation: the second virial "
    "coefficient of eq-md-b2 accounts for most, but honestly not all, "
    "of the departure at rho = 0.10 — the rest is the third",
    f"measured/B_2 share = {b2_share:.3f} +- {share_sigma:.3f} at "
    f"rho = {rho_s[idx_10]:.2f}: significantly positive, and not above B_2's own "
    "prediction, which is as sharp as this many samples allow",
)

## Notebook summary

- The 12-6 potential was differentiated into the pair force
  {eq}`eq-md-force`, written from powers of $r^{-2}$ with no square
  root, and certified against a central difference of its own
  potential at $r = 1.2$, $1.5$ and $2.0$; the well bottoms out at
  exactly $-\varepsilon$ at $r_{\min} = 2^{1/6}$, and still carries
  $u(r_c) = -0.0163\,\varepsilon$ at the cutoff.
- The minimum-image convention {eq}`eq-md-minimage` reproduced an
  exhaustive search over all nine periodic images *exactly*, on 500
  random pairs, and the assembled force routine was certified twice
  over: $\mathbf F = -\nabla U$ by central differences on one atom,
  and $W = -\mathrm{d}U/\mathrm{d}\lambda$ under a uniform scaling of
  coordinates and box together.
- The two boundary decisions were priced by removing them. Wrapping
  positions but taking raw coordinate differences drove the energy
  past $Nk_BT$ within one reduced time unit; truncating without
  shifting left the trajectory bit-for-bit identical and multiplied
  the energy excursion by nineteen, a drift with no integrator to
  blame.
- Velocity rescaling was measured against the ensemble it is often
  mistaken for: it holds $\sigma_K/\langle K\rangle$ at
  $2\times10^{-16}$ where the canonical value {eq}`eq-md-kfluct` is
  $0.126$, and a run rescaled to exactly $T = 1.000$ and then released
  settled at $\langle T\rangle = 0.947$.
- A gas started with every atom at one speed became Maxwell-Boltzmann
  under smooth, deterministic, time-reversible flow: the
  parameter-free ratio $\langle v^4\rangle/\langle v^2\rangle^2$ went
  from $1.00$ to $2.05$ against the Rayleigh value $2$, the relative
  speed width from $0.01$ to $0.53$ against $\sqrt{4/\pi-1}=0.523$,
  and a Kolmogorov-Smirnov test that rejects the initial shell at
  $p = 4\times10^{-3}$ fails to reject the relaxed gas at $p = 0.58$.
- Correlation was priced: $\tau_{\rm int} = 29.8$ samples for the
  temperature against $4.2$ for the pressure, a blocked bar $7.9$
  times the naive $\sigma/\sqrt n$ where {eq}`eq-md-tau` predicts
  $\sqrt{2\tau_{\rm int}} = 7.7$, and $4001$ samples worth $67$
  independent ones.
- And the pressure was measured from the trajectory for the first time
  in this course: ideal to $4\%$ at $\rho = 0.02$, below ideal out to
  $\rho = 0.35$ where attraction wins, $4.7$ times ideal at
  $\rho = 0.80$ where the core does, with the low-density slope
  predicted independently by the quadrature $B_2(1) = -0.818\,\sigma^2$
  of {eq}`eq-md-b2`.

## Outlook

- **Structure, thermostats, melting, and three dimensions.** All four
  are waiting, already built, in the sister course *Molecular &
  Materials Modelling*: its notebook
  [Molecular Dynamics of Lennard-Jones Clusters](https://ramador09.github.io/molecular-materials-modelling-public/notebooks/03-molecular-dynamics/lennard-jones-md.html)
  runs a three-dimensional argon cluster, computes the radial
  distribution function $g(r)$ and reads the coordination number off
  its first shell, thermostats the run, and melts it. Deliberately not
  repeated here: this notebook is the boundary layer, that one is the
  structure.
- **A real thermostat.** Exercise 6 said what rescaling is not.
  Nosé-Hoover extends the Hamiltonian with a thermal degree of
  freedom and provably samples the canonical distribution; canonical
  velocity rescaling adds exactly the missing fluctuation as noise;
  and the Langevin BAOAB integrator of
  [§5.11](taste-of-nonequilibrium.ipynb), applied to the interacting
  system rather than to free particles, is a third route. Constant
  pressure needs a barostat and a fluctuating box on top.
- **Cheaper neighbours.** The $O(N^2)$ table wastes almost all of its
  work beyond $r_c$; Verlet neighbour lists and cell lists reduce it
  to $O(N)$ and are what make a million-atom simulation possible. The
  long-range electrostatics that a charged system needs cannot be cut
  off at all, and Ewald summation splitting the sum between real and
  reciprocal space is the standard answer.
- **Where the model comes from.** The 12-6 form is a convenient guess,
  and its exponent $12$ has no physics in it. Modern simulation
  replaces it by forces computed from electronic structure, either
  directly or through machine-learned interatomic potentials fitted
  to them, and the whole apparatus built here (periodic boundaries,
  minimum image, cutoffs, virial pressure, blocked error bars)
  survives that replacement untouched. Only the line that returns
  $u$ and $f/r$ changes.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()